# RetroRestore — Google Colab'da çalıştır

Bu not defteri projeyi **hiçbir şey kurmadan**, ücretsiz GPU ile çalıştırır ve size web sitesinin linkini verir.

1. Menüden **Çalışma zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin (yoksa CPU ile de çalışır, daha yavaş).
2. **Çalışma zamanı → Tümünü çalıştır** (Ctrl+F9).
3. En alttaki hücrede çıkan **linke tıklayın** → fotoğraf/video yükleyip deneyin.


In [ ]:
# 1) Projeyi indir ve modelleri hazırla (~1-2 dk)
import os, subprocess, sys
if not os.path.exists('/content/retro-restore'):
    !git clone -q https://github.com/aysecetin0/retro-restore.git /content/retro-restore
else:
    !cd /content/retro-restore && git pull -q
%cd /content/retro-restore
!pip -q install flask
!python scripts/download_models.py
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK (CPU kullanılacak)')


In [ ]:
# 2) (İsteğe bağlı) Kurulumu doğrula: testler + gerçek Real-ESRGAN çıkarımı
!python -m pytest -q -p no:cacheprovider 2>&1 | tail -2
!python enhance_cli.py dataset/degraded/realworld_x4/astronaut.png -o /content/test_sr.png --scale 4 --model realesr-general-x4v3
from IPython.display import Image, display
display(Image('/content/test_sr.png', width=400))


In [ ]:
# 3) Web sitesini başlat ve linki göster
import time
subprocess.run('pkill -f run.py', shell=True)
log = open('/content/server.log', 'w')
proc = subprocess.Popen([sys.executable, 'run.py', '--host', '0.0.0.0', '--port', '8000'], stdout=log, stderr=subprocess.STDOUT)
time.sleep(6)
print(open('/content/server.log').read()[-800:])
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
from IPython.display import HTML
display(HTML(f'<h2>👉 <a href="{url}" target="_blank">RetroRestore web sitesini aç</a></h2><p>{url}</p>'))


In [ ]:
# 4) (İsteğe bağlı) Veri setinde toplu değerlendirme → dataset/results (rapordaki tablo ve ızgaralar)
!python scripts/evaluate.py
print(open('dataset/results/summary.md').read())
